# FHIR Encounter Silver Lakeflow Transformation

## Purpose

I use this notebook as the transformation source for the FHIR Silver Lakeflow
pipeline.

I keep the Encounter Bronze-to-Silver transformation logic that I previously
developed and tested, while Lakeflow now manages the Silver output and applies
the approved Encounter quality contract during refresh.

### Source

`health_insurance.bronze.fhir_encounter_raw`

### Pipeline target

`health_insurance.silver.fhir_encounter`

### Production changes

In this pipeline version:

- I keep the null-safe FHIR parsing and conformance logic I already validated.
- I use an explicit minimal FHIR Encounter schema instead of inferring the
  schema at runtime.
- I retain FHIR `meta.versionId` and `meta.lastUpdated` as technical metadata.
- I keep the latest available version of each Encounter resource.
- I load WARN, DROP, and FAIL rules dynamically from Unity Catalog.
- I let Lakeflow manage Silver persistence and quality metrics.

The FHIR Bronze table is cumulative and incrementally populated by Auto Loader.
I therefore use a Lakeflow materialized view over the current Bronze state so I
can resolve the latest Encounter version deterministically.


In [0]:
# importing the Lakeflow API, Spark functions, schemas, and window support.

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_encounter_raw"
QUALITY_RULES_TABLE = f"{CATALOG}.governance.quality_rules"


In [0]:
# loading the active Encounter quality contract from Unity Catalog.

def get_quality_rules(dataset, severity):
    rows = (
        spark.read
        .table(QUALITY_RULES_TABLE)
        .filter(
            (F.col("dataset") == dataset)
            & (F.col("severity") == severity)
            & F.col("is_active")
        )
        .select(
            "rule_name",
            "constraint"
        )
        .collect()
    )

    return {
        row["rule_name"]: row["constraint"]
        for row in rows
    }


ENCOUNTER_WARN_RULES = get_quality_rules("encounter", "WARN")
ENCOUNTER_DROP_RULES = get_quality_rules("encounter", "DROP")
ENCOUNTER_FAIL_RULES = get_quality_rules("encounter", "FAIL")


## Explicit FHIR Encounter schema

During development I inferred the combined Encounter schema from Bronze.

For the pipeline version, I define only the FHIR structures that my Silver
transformation consumes. This keeps the production transformation deterministic
while allowing optional FHIR fields to remain NULL.

I also include the FHIR metadata required for latest-version handling.


In [0]:
# defining the minimal FHIR Encounter schema required by this transformation.

coding_schema = T.StructType([
    T.StructField("system", T.StringType(), True),
    T.StructField("code", T.StringType(), True),
    T.StructField("display", T.StringType(), True)
])

codeable_concept_schema = T.StructType([
    T.StructField(
        "coding",
        T.ArrayType(coding_schema),
        True
    )
])

reference_schema = T.StructType([
    T.StructField("reference", T.StringType(), True)
])

participant_schema = T.StructType([
    T.StructField(
        "individual",
        reference_schema,
        True
    )
])

period_schema = T.StructType([
    T.StructField("start", T.StringType(), True),
    T.StructField("end", T.StringType(), True)
])

class_schema = T.StructType([
    T.StructField("system", T.StringType(), True),
    T.StructField("code", T.StringType(), True),
    T.StructField("display", T.StringType(), True)
])

meta_schema = T.StructType([
    T.StructField("versionId", T.StringType(), True),
    T.StructField("lastUpdated", T.StringType(), True)
])

encounter_schema = T.StructType([
    T.StructField("id", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("class", class_schema, True),
    T.StructField(
        "type",
        T.ArrayType(codeable_concept_schema),
        True
    ),
    T.StructField(
        "subject",
        reference_schema,
        True
    ),
    T.StructField(
        "participant",
        T.ArrayType(participant_schema),
        True
    ),
    T.StructField(
        "period",
        period_schema,
        True
    ),
    T.StructField(
        "serviceProvider",
        reference_schema,
        True
    ),
    T.StructField("meta", meta_schema, True)
])


## Reusable Encounter transformation

I keep the transformation itself separate from the Lakeflow dataset definition.

The function performs the same Encounter parsing, null-safe type and participant
selection, reference parsing, timestamp conversion, duration calculation, and
status/class standardization I validated in the development notebook.

I additionally retain FHIR version metadata and resolve the latest resource
version before publishing Silver.


In [0]:
# applying the validated FHIR Encounter Bronze-to-Silver transformation.

def transform_encounter(encounter_bronze_df):

    encounter_parsed_df = (
        encounter_bronze_df
        .withColumn(
            "_record_hash",
            F.sha2(F.col("raw_json"), 256)
        )
        .withColumn(
            "encounter",
            F.from_json(
                F.col("raw_json"),
                encounter_schema
            )
        )
    )

    encounter_core_df = (
        encounter_parsed_df
        .select(
            F.col("encounter.id").alias("encounter_id"),
            F.col("encounter.status").alias("status"),

            F.col("encounter.class").alias(
                "encounter_class_struct"
            ),
            F.col("encounter.type").alias(
                "encounter_type_array"
            ),

            F.col("encounter.subject.reference").alias(
                "patient_reference"
            ),
            F.col("encounter.participant").alias(
                "participant_array"
            ),

            F.col("encounter.period.start").alias(
                "start_datetime_raw"
            ),
            F.col("encounter.period.end").alias(
                "end_datetime_raw"
            ),

            F.col("encounter.serviceProvider.reference").alias(
                "organization_reference"
            ),

            F.col("encounter.meta.versionId").alias(
                "fhir_version_id"
            ),
            F.to_timestamp(
                F.col("encounter.meta.lastUpdated")
            ).alias("fhir_last_updated"),

            "_record_hash",
            "_ingested_at",
            "_source_system",
            "_resource_type"
        )
    )

    encounter_class_df = (
        encounter_core_df

        .withColumn(
            "encounter_class",
            F.col("encounter_class_struct.code")
        )

        .withColumn(
            "encounter_class_system",
            F.col("encounter_class_struct.system")
        )
    )

    encounter_type_df = (
        encounter_class_df

        .withColumn(
            "primary_type",
            F.expr("get(encounter_type_array, 0)")
        )

        .withColumn(
            "encounter_type_code",
            F.expr(
                "get(primary_type.coding.code, 0)"
            )
        )

        .withColumn(
            "encounter_type",
            F.expr(
                "get(primary_type.coding.display, 0)"
            )
        )

        .withColumn(
            "encounter_type_system",
            F.expr(
                "get(primary_type.coding.system, 0)"
            )
        )
    )

    encounter_participant_df = (
        encounter_type_df

        .withColumn(
            "primary_participant",
            F.expr("get(participant_array, 0)")
        )

        .withColumn(
            "practitioner_reference",
            F.col(
                "primary_participant.individual.reference"
            )
        )
    )

    encounter_refs_df = (
        encounter_participant_df

        .withColumn(
            "patient_id",
            F.when(
                F.col("patient_reference").startswith("Patient/"),
                F.regexp_extract(
                    F.col("patient_reference"),
                    r"^Patient/(.+)$",
                    1
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .withColumn(
            "practitioner_id",
            F.when(
                F.col("practitioner_reference").startswith("Practitioner/"),
                F.regexp_extract(
                    F.col("practitioner_reference"),
                    r"^Practitioner/(.+)$",
                    1
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .withColumn(
            "organization_id",
            F.when(
                F.col("organization_reference").startswith("Organization/"),
                F.regexp_extract(
                    F.col("organization_reference"),
                    r"^Organization/(.+)$",
                    1
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )

    encounter_typed_df = (
        encounter_refs_df

        .withColumn(
            "start_datetime",
            F.to_timestamp("start_datetime_raw")
        )

        .withColumn(
            "end_datetime",
            F.to_timestamp("end_datetime_raw")
        )
    )

    encounter_enriched_df = (
        encounter_typed_df

        .withColumn(
            "encounter_duration_minutes",
            F.when(
                F.col("start_datetime").isNotNull()
                & F.col("end_datetime").isNotNull(),

                (
                    F.unix_timestamp("end_datetime")
                    - F.unix_timestamp("start_datetime")
                ) / 60
            )
            .otherwise(
                F.lit(None)
            )
            .cast("int")
        )
    )

    encounter_standardized_df = (
        encounter_enriched_df

        .withColumn(
            "status",
            F.upper(F.trim("status"))
        )

        .withColumn(
            "encounter_class",
            F.upper(F.trim("encounter_class"))
        )
    )

    encounter_conformed_df = (
        encounter_standardized_df

        .select(
            "encounter_id",
            "patient_id",
            "practitioner_id",
            "organization_id",

            "status",

            "encounter_class",
            "encounter_class_system",

            "encounter_type_code",
            "encounter_type",
            "encounter_type_system",

            "start_datetime",
            "end_datetime",
            "encounter_duration_minutes",

            "fhir_version_id",
            "fhir_last_updated",

            "_record_hash",
            "_source_system",
            "_resource_type",
            "_ingested_at"
        )

        .withColumn(
            "_silver_transformed_at",
            F.current_timestamp()
        )
    )

    # I am using the FHIR Encounter ID as the main deduplication key.
    # If the resource ID is missing, I use the raw payload hash so malformed
    # records remain distinct until the DROP expectations evaluate them.

    encounter_versioned_df = (
        encounter_conformed_df
        .withColumn(
            "_dedup_key",
            F.coalesce(
                F.col("encounter_id"),
                F.col("_record_hash")
            )
        )
    )

    latest_encounter_window = (
        Window
        .partitionBy("_dedup_key")
        .orderBy(
            F.col("fhir_last_updated").desc_nulls_last(),
            F.col("_ingested_at").desc_nulls_last(),
            F.col("fhir_version_id").desc_nulls_last()
        )
    )

    encounter_latest_df = (
        encounter_versioned_df

        .withColumn(
            "_version_rank",
            F.row_number().over(latest_encounter_window)
        )

        .filter(
            F.col("_version_rank") == 1
        )

        .drop(
            "_version_rank",
            "_dedup_key",
            "_record_hash"
        )
    )

    return encounter_latest_df


## Lakeflow-managed Encounter Silver dataset

I attach the centrally governed Encounter expectations directly to the Silver
materialized view.

- WARN rules preserve the record and expose quality metrics.
- DROP rules remove unusable Encounter records from validated Silver.
- FAIL rules stop the Encounter flow when a critical contract is violated.

The materialized view reads the cumulative Bronze state, resolves the latest
FHIR Encounter version, and publishes one current Silver record per Encounter.


In [0]:
# defining the pipeline-managed FHIR Encounter Silver materialized view.

@dp.materialized_view(
    name="fhir_encounter",
    comment="Validated and latest-version FHIR Encounter records."
)
@dp.expect_all(ENCOUNTER_WARN_RULES)
@dp.expect_all_or_drop(ENCOUNTER_DROP_RULES)
@dp.expect_all_or_fail(ENCOUNTER_FAIL_RULES)
def fhir_encounter():

    encounter_bronze_df = spark.read.table(
        SOURCE_TABLE
    )

    return transform_encounter(
        encounter_bronze_df
    )


## Pipeline result

This notebook no longer writes `health_insurance.silver.fhir_encounter`
manually.

When I add it as a source to the FHIR Silver Lakeflow pipeline:

1. Lakeflow reads the cumulative FHIR Encounter Bronze table.
2. I parse the Encounter JSON with an explicit production schema.
3. I apply the previously validated Encounter transformation logic.
4. I retain FHIR version metadata and select the latest resource version.
5. Lakeflow evaluates the active governed Encounter quality rules.
6. Lakeflow manages the Silver materialized view and expectation metrics.

The development-only schema inference, `display()`, profiling counts, direct
Delta overwrite, row reconciliation, and post-write verification cells are no
longer part of the pipeline execution path.
